In [171]:
# key realization: we don't have to spend all of our money!

import numpy as np

def rs_given_speed(speed_stat: float, v: int):
    best_r_s = [0, 0]
    best_profit = -1
    for r in range(100 - v):
        for s in range(100 - v - r):
            profit = 200000 * np.log(1 + r) / np.log(101) * 0.07 * s * speed_stat
            if profit > best_profit:
                best_r_s = [r, s]
                best_profit = profit
    return best_r_s

In [172]:
# generate prior of users
# 60% - 70%: balanced
#  2% -  4%: research (these guys are stupid)
#  8% - 12%: scale (same here)
# 20% - 25%: speedy as f
#  2% -  4%: random (they're not very smart lol)

# assumptions:
# all people will use between 90% - 100% of their money, vast majority using all of it
# base spreads across different types of people

percentage_probs = [0.8 * 0.2 ** (10 - i) for i in range(11)]
sum_pp = sum(percentage_probs)
for i in range(len(percentage_probs)):
    percentage_probs[i] /= sum_pp

def generate_deviation():
    deviation_spread = np.random.choice(list(range(-10, 11)), size=3)
    while deviation_spread.sum() != 0:
        deviation_spread = np.random.choice(list(range(-10, 11)), size=3)
    return deviation_spread

def generate_trader(idx: int):
    # allocate %age of investment
    percentages = list(range(90, 101, 1))
    total_alloc = np.random.choice(percentages, p=percentage_probs)

    base_spreads = [np.array([15, 50, 35]), 
                    np.array([45, 20, 35]), 
                    np.array([10, 60, 30]), 
                    np.array([10, 30, 60]), 
                    np.array([30, 30, 40])]
    base_spread = base_spreads[idx]
    for _ in range(100 - total_alloc):
        cut_idx = np.random.choice(3)
        while base_spread[cut_idx] == 0:
            cut_idx = np.random.choice(3)
        base_spread[cut_idx] -= 1

    deviation_spread = generate_deviation()
    final_spread = base_spread + deviation_spread
    while not (final_spread[0] >= 0 and final_spread[1] >= 0 and final_spread[2] >= 0):
        deviation_spread = generate_deviation()
        final_spread = base_spread + deviation_spread
    return final_spread

In [173]:
# simulate the final outcome

def get_trader_ranks(traders: list[np.ndarray]):
    scale_dict = {}
    for trader in traders:
        scale = int(trader[2])
        if scale not in scale_dict:
            scale_dict[scale] = 0
        scale_dict[scale] += 1
    
    scale_list = []
    for key, val in scale_dict.items():
        scale_list.append([key, val])
    scale_list.sort(key = lambda x: -x[0])

    cur_rank, max_rank = 1, 1
    ranks = {}
    for l in scale_list:
        ranks[l[0]] = cur_rank
        max_rank = cur_rank
        cur_rank += l[1]
    return ranks, max_rank

def calculate_trader_scores(traders: list[np.ndarray]):
    ranks, max_rank = get_trader_ranks(traders)
    slope = -0.8 / (max_rank - 1)

    # 1 --> 0.9; mr --> 0.1
    # slope = (0.1 - 0.9) / (mr - 1)
    # y - 0.9 = slope (x - 1)
    # y = (0.9 - slope) + slope * x

    result = []
    for trader in traders:
        if not (trader[0] >= 0 and trader[1] >= 0 and trader[2] >= 0):
            raise Exception(f"ERROR: {trader[0]}, {trader[1]}, {trader[2]} has a neg")
        research = 200000 * np.log(1 + trader[0]) / np.log(1 + 100)
        scale = 7 * trader[1] / 100
        speed = (0.9 - slope) + slope * ranks[trader[2]]
        # print(research, "\t", scale, "\t", speed)
        if not (0.1 <= speed <= 0.9):
            speed = max(min(speed, 0.9), 0.1) # PYTHON IS KHOOOOOOSE

        # error handling
        if research < 0:
            print(f"ERROR: trader[0] = {trader[0]}")
        assert(research >= 0)
        assert(0.1 <= speed <= 0.9)
        
        result.append(research * scale * speed - 50000 * trader.sum() / 100)
    return result

In [174]:
# find optimal traders for different populations, see what they look like

from copy import deepcopy

def generate_traders(trader_cnt: int):
    traders = []
    for _ in range(trader_cnt):
        people_dist = [np.random.uniform(0.60, 0.70),
                       np.random.uniform(0.02, 0.04),
                       np.random.uniform(0.08, 0.12),
                       np.random.uniform(0.20, 0.25),
                       np.random.uniform(0.02, 0.04)]
        sum_pd = sum(people_dist)
        for i in range(5):
            people_dist[i] /= sum_pd
        trader_idx = np.random.choice(5, p=people_dist)
        traders.append(generate_trader(trader_idx))
    return traders

def find_optimal_trader(traders: list[np.ndarray]):
    best_profit = -1e9
    best_trader = np.ndarray([0, 0, 0])

    for v in range(100):
        traders_copy = deepcopy(traders)
        traders_copy.append([0, 0, v])
        ranks, _ = get_trader_ranks(traders_copy)
        speed_stat = ranks[v]

        rs = rs_given_speed(speed_stat, v)
        r = rs[0]
        s = rs[1]
        cur_trader = np.array([r, s, v])
        traders_copy.pop()
        traders_copy.append(cur_trader)
        cur_trader_profit = calculate_trader_scores(traders_copy)[-1]

        if cur_trader_profit > best_profit:
            best_profit = cur_trader_profit
            best_trader = cur_trader
    
    return best_trader, best_profit

In [ ]:
# tester code

N = 1000
T = 30
best_traders = np.array([])
best_profits = np.array([])

for _ in range(T):
    traders = generate_traders(N - 1)
    best_trader, best_profit = find_optimal_trader(traders)
    best_profit = best_profit.round(2)
    best_traders = np.append(best_traders, best_trader)
    best_profits = np.append(best_profits, best_profit)
    print(best_trader, '\t', best_profit)

best_traders = best_traders.reshape(-1, 3)
print("average best trader:", best_traders.mean(axis=0).round(2))
print("average best profit:", best_profits.mean().round(2))

# reflections:
# the model (and its many iterations) REALLY likes the following + similar strats:
# [12, 35, 52], [14, 41, 44] (current), [15, 42, 42], [16, 46, 37]
# if we do pick either one, it can help to sacrifice a little scale for speed --> [14, 40, 45]

[14 41 44] 	 194501.96
[14 42 43] 	 194645.18
[14 41 44] 	 187482.26
[14 40 45] 	 185856.16
[14 41 44] 	 190929.28
[14 42 43] 	 190773.16
[14 40 45] 	 189054.42
[14 41 44] 	 190929.28
[14 41 44] 	 191824.35
[14 41 44] 	 194021.71
[14 41 44] 	 183963.18
[14 41 44] 	 188087.91
[14 42 43] 	 191878.89
[14 40 45] 	 192292.39
[14 42 43] 	 187804.21
[14 40 45] 	 186503.32
[14 41 44] 	 190803.25
[14 40 45] 	 186443.47
[14 40 45] 	 182300.22
[14 41 44] 	 188292.23
[14 41 44] 	 192698.87
[14 40 45] 	 189077.5
[14 40 45] 	 185795.54
[14 42 43] 	 193815.46
[14 40 45] 	 180186.65
[14 41 44] 	 188022.24
[14 40 45] 	 190226.15
[14 41 44] 	 187415.92
[14 40 45] 	 187233.68
[14 40 45] 	 186119.83
average best trader: [14.   40.77 44.23]
average best profit: 188965.96
